In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
import joblib


In [2]:
df = pd.read_csv("../data/processed/cleaned_churn.csv")
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.shape

(7043, 20)

In [4]:
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

In [5]:
df.dtypes

gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object

In [6]:
X = df.drop(columns=["Churn"])
y= df["Churn"]


In [7]:
y = y.map({
    "No":0,
    "Yes":1
})
y.head()

0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int64

In [8]:
y.value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [9]:
numeric_feature = X.select_dtypes(include=["int", "float"]).columns.tolist()
numeric_feature

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

In [10]:

categorical_feature=X.select_dtypes(
    include="object"
).columns.tolist()
categorical_feature

C:\Users\Hp\AppData\Local\Temp\ipykernel_3800\591929473.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_feature=X.select_dtypes(


['gender',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod']

In [11]:
X_train, X_test, y_train, y_test=train_test_split(X, y, random_state=42, test_size=0.2)

In [12]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(5634, 19)
(1409, 19)
(5634,)
(1409,)


In [13]:
print("Training Traget Mean:", y_train.mean())
print("Testing Target Mean:",y_test.mean())
print("\nTraining Target Median:", y_train.median())
print(y_test.median())

Training Traget Mean: 0.2655307064252751
Testing Target Mean: 0.26472675656493966

Training Target Median: 0.0
0.0


In [14]:
X.isnull().sum().sum()

np.int64(0)

In [15]:
y.isnull().sum()

np.int64(0)

In [16]:
numeric_transformer= StandardScaler()


In [17]:
categorical_transformer=OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)


In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_feature
                   
        ),
        (
            "cat",
            categorical_transformer,
            categorical_feature
        )
     ]
        
)

In [19]:
X_train_processed = preprocessor.fit_transform(X_train)

In [20]:
X_test_processed = preprocessor.transform(X_test)

In [21]:
print("Before:",X_train.shape)
print("After:", X_train_processed.shape)
print("Before:", X_test.shape)
print("After:", X_test_processed.shape)


Before: (5634, 19)
After: (5634, 45)
Before: (1409, 19)
After: (1409, 45)


In [22]:
feature_names = preprocessor.get_feature_names_out()
print(feature_names)

['num__SeniorCitizen' 'num__tenure' 'num__MonthlyCharges'
 'num__TotalCharges' 'cat__gender_Female' 'cat__gender_Male'
 'cat__Partner_No' 'cat__Partner_Yes' 'cat__Dependents_No'
 'cat__Dependents_Yes' 'cat__PhoneService_No' 'cat__PhoneService_Yes'
 'cat__MultipleLines_No' 'cat__MultipleLines_No phone service'
 'cat__MultipleLines_Yes' 'cat__InternetService_DSL'
 'cat__InternetService_Fiber optic' 'cat__InternetService_No'
 'cat__OnlineSecurity_No' 'cat__OnlineSecurity_No internet service'
 'cat__OnlineSecurity_Yes' 'cat__OnlineBackup_No'
 'cat__OnlineBackup_No internet service' 'cat__OnlineBackup_Yes'
 'cat__DeviceProtection_No' 'cat__DeviceProtection_No internet service'
 'cat__DeviceProtection_Yes' 'cat__TechSupport_No'
 'cat__TechSupport_No internet service' 'cat__TechSupport_Yes'
 'cat__StreamingTV_No' 'cat__StreamingTV_No internet service'
 'cat__StreamingTV_Yes' 'cat__StreamingMovies_No'
 'cat__StreamingMovies_No internet service' 'cat__StreamingMovies_Yes'
 'cat__Contract_Month-

In [23]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names)
X_train_processed_df.head()
X_train_processed_df.shape

(5634, 45)

In [24]:
X_test_processed_df=pd.DataFrame(
    X_test_processed,
    columns=feature_names
)
X_test_processed_df.head()
X_test_processed_df.shape


(1409, 45)

In [25]:
X_train_processed_df.columns.equals(X_test_processed_df.columns)

True

In [26]:
X_train_processed_df.isnull().sum().sum()

np.int64(0)

In [27]:
X_test_processed_df.isnull().sum().sum()

np.int64(0)

In [28]:
X_train_processed_df.to_csv("../data/processed/X_train_processed.csv", index=False)


In [29]:
X_test_processed_df.to_csv("../data/processed/X_test_processed.csv", index=False)

In [30]:
y_train.to_csv("../data/processed/y_train.csv", index=False)

In [31]:
y_test.to_csv("../data/processed/y_test.csv", index=False)

In [32]:
joblib.dump(
    preprocessor,
    "../models/preprocessor.pkl"
)

['../models/preprocessor.pkl']

In [33]:
import os
os.path.exists("../models/preprocessor.pkl") 

True